# TeaLeafDiseaseBD — external-dataset test (Reviewer 1.2, 2.1, 2.2) — v3

Tests the 70-label screener, **unchanged**, on two independently collected tea datasets:
**East West University (EWU)** (Roboflow YOLO export, leaf boxes) and **Allynugger** (folders per class).
Nothing is trained or tuned on external data for the zero-shot results; the decision threshold is fixed
at 95% sensitivity on the TeaLeafDiseaseBD test split *before* any external image is scored.

Models
1. **Frozen probe (teacher)**: refit exactly from `features_dinov1_crop.npz` (5 seed draws, verified).
2. **Deployed student `tea_screener.onnx`**: the file exported by `revision_v1_reproduce.ipynb`
   (checked against `deploy/export_report.json` by SHA-256 before use).

Analyses
- zero-shot at the fixed in-distribution threshold (probe and deployed student)
- label-free threshold from the target set's healthy prevalence (probe and deployed student)
- local recalibration of the probe with 1/2/5/10 labels per target class, near-duplicate groups held out

Built-in checks: preprocessing reproduces cached features (Cell 3); probes reproduce published AUCs
(Cell 4); ONNX checksum and test AUC match the export report (Cell 11); contact sheets (Cell 8);
leakage against TeaLeafDiseaseBD (Cell 9); within-set near-duplicate rates (Cell 13).

Runtime on a GPU runtime: ~10–15 min with feature caches from an earlier run, ~25 min from scratch.

In [ ]:
# CELL 0 — CONFIG  (edit paths only)
DRIVE = '/content/drive/MyDrive'
TEA_ROOT   = f'{DRIVE}/Tea Leaf Dataset'                      # TeaLeafDiseaseBD, one folder per class
RES        = f'{DRIVE}/TeaLeaf_DINOv2_Study/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
DEPLOY     = f'{RES}/deploy'                                  # written by revision_v1_reproduce.ipynb
ONNX_PATH  = f'{DEPLOY}/tea_screener.onnx'
ONNX_REPORT= f'{DEPLOY}/export_report.json'
ONNX_ID_SCORES = f'{DEPLOY}/tea_screener_test_scores.npy'     # test-split scores saved at export time
EWU_DIR    = f'{DRIVE}/East West University dataset'          # data.yaml + Roboflow zip
ALLY_DIR   = f'{DRIVE}/Allynugger'                            # Anthracnose / Grey Blight / Healthy
OUT        = f'{RES}/external_test'
SEED, N_SHOT, N_REP, PAD, IMG, JPEG_Q = 42, 10, 5, 0.06, 224, 95
HEALTHY_IDX = 3
TARGET_SENS = 0.95
RUN_ALLY_OTSU = True      # Otsu on Allynugger returns ~the whole frame (median 99.8%); False saves ~2 min
DUP = 0.95                # cosine threshold for near-duplicate groups in the recalibration
RECAL_DRAWS = 40          # draws per label budget (5 draws vary by about ±0.006 AUC at 10/class)


In [ ]:
# CELL 1 — setup
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime pyyaml
import os, io, re, glob, json, zipfile, random, warnings, shutil, time, hashlib
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from collections import Counter
import numpy as np, cv2, yaml
from PIL import Image
from scipy import ndimage
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
os.makedirs(OUT, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')


In [ ]:
# CELL 2 — preprocessing (same recipe as the paper: Otsu on HSV V|S, largest component,
#           fill holes, bbox crop with 6% padding, resize 224x224, JPEG q95 round-trip)
def leaf_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0)
    lab, n = ndimage.label(m)
    if n == 0:
        return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def box_of(mask):
    ys, xs = np.where(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

def crop_box(rgb, box, pad=PAD, size=IMG):
    H, W = rgb.shape[:2]; x0, y0, x1, y1 = box
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0 = int(max(0, x0 - px)), int(max(0, y0 - py))
    x1, y1 = int(min(W, x1 + px)), int(min(H, y1 + py))
    im = Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q)       # match the cached-crop pipeline
    return Image.open(io.BytesIO(buf.getvalue())).convert('RGB')

def load_rgb(p):
    return np.array(Image.open(p).convert('RGB'))

def otsu_crop(rgb):
    m = leaf_mask(rgb)
    return crop_box(rgb, box_of(m)), float(m.mean())

def whole_crop(rgb):
    H, W = rgb.shape[:2]
    return crop_box(rgb, (0, 0, W, H), pad=0.0), 1.0


In [ ]:
# CELL 3 — DINO ViT-S/16 + feature extractor, then SANITY CHECK against cached features
import torch, torchvision.transforms as T
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(dev).eval()
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

@torch.no_grad()
def feats(pil_list, bs=64):
    out = []
    for i in range(0, len(pil_list), bs):
        x = torch.stack([TO_T(p) for p in pil_list[i:i + bs]]).to(dev)
        f = dino(x).float()
        out.append(torch.nn.functional.normalize(f, dim=1).cpu().numpy())
    return np.concatenate(out)

F = np.load(FEATS); Xtr, ytr, Xte, yte, dte = F['Xtr'], F['ytr'], F['Xte'], F['yte'], F['dte']
SPL = json.load(open(SPLIT)); TEST_PATHS = SPL['test_paths']
local = lambda p: os.path.join(TEA_ROOT, *p.split('/')[-2:])
rng = np.random.default_rng(0); chk = rng.choice(len(TEST_PATHS), 40, replace=False)
fx = feats([otsu_crop(load_rgb(local(TEST_PATHS[i])))[0] for i in chk])
cos = (fx * Xte[chk]).sum(1)
print('cosine(our preprocessing, cached features): median %.4f  min %.4f' % (np.median(cos), cos.min()))
if np.median(cos) < 0.98:
    print('WARNING: preprocessing does NOT reproduce the training features. Stop and report this '
          'before using any external number (crop/pad/resize rule differs from the cache builder).')
else:
    print('OK: preprocessing matches the training pipeline.')


In [ ]:
# CELL 4 — refit the 5 exact probes; fix the 95%-sensitivity threshold IN-DISTRIBUTION
J = json.load(open(EJSON))
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(7)])
PROBES, THR = [], []
for rep in range(N_REP):
    idx = draw(rep)
    m = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr[idx], ytr[idx])
    s = 1 - m.predict_proba(Xte)[:, HEALTHY_IDX]
    pos = np.sort(s[dte == 1]); THR.append(float(pos[int(np.floor((1 - TARGET_SENS) * len(pos)))]))
    PROBES.append(m)
aucs = [roc_auc_score(dte, 1 - m.predict_proba(Xte)[:, HEALTHY_IDX]) for m in PROBES]
print('probe AUC per rep  ', np.round(aucs, 4))
print('published per rep  ', np.round(J['probe_auc_per_rep'], 4))
assert np.allclose(aucs, J['probe_auc_per_rep'], atol=1e-3), 'probe did not reproduce — stop'
print('95%-sensitivity thresholds (fixed before external data):', np.round(THR, 4))


In [ ]:
# CELL 5 — EWU: unzip Roboflow export, read YOLO boxes, drop Roboflow augmented copies
EWU_X = '/content/ewu'
if not os.path.exists(EWU_X):
    z = [p for p in glob.glob(f'{EWU_DIR}/*.zip')]
    assert z, 'no zip found in ' + EWU_DIR
    zipfile.ZipFile(z[0]).extractall(EWU_X)
yml = glob.glob(f'{EWU_X}/**/data.yaml', recursive=True) or glob.glob(f'{EWU_DIR}/data.yaml')
names = yaml.safe_load(open(yml[0]))['names']
names = list(names.values()) if isinstance(names, dict) else list(names)
print('EWU classes:', names)

def rf_stem(fname):          # Roboflow: <orig>_jpg.rf.<hash>.jpg  -> <orig>
    m = re.match(r'^(.*?)_(jpe?g|png|bmp|webp)\.rf\.', os.path.basename(fname), re.I)
    return m.group(1) if m else os.path.splitext(os.path.basename(fname))[0]

def split_rank(p):           # prefer un-augmented splits when de-duplicating
    return 0 if '/test/' in p else 1 if '/valid/' in p else 2

imgs = sorted([p for p in glob.glob(f'{EWU_X}/**/*', recursive=True) if p.lower().endswith(IMG_EXT)],
              key=lambda p: (split_rank(p), p))   # deterministic: same copy kept every run
seen, EWU = set(), []
for p in imgs:
    st = rf_stem(p)
    if st in seen: continue
    lbl = re.sub(r'/images/', '/labels/', p); lbl = os.path.splitext(lbl)[0] + '.txt'
    if not os.path.exists(lbl): continue
    seen.add(st)
    for line in open(lbl).read().strip().splitlines():
        v = line.split()
        if len(v) != 5: continue          # skip polygon rows
        EWU.append(dict(path=p, cls=names[int(v[0])], yolo=tuple(map(float, v[1:]))))
print('EWU: %d unique source images, %d leaf boxes (of %d image files incl. augmentations)'
      % (len(seen), len(EWU), len(imgs)))
from collections import Counter; print(Counter(r['cls'] for r in EWU))

def yolo_to_box(yolo, W, H):
    cx, cy, w, h = yolo
    return (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H


In [ ]:
# CELL 6 — Allynugger: one folder per class, copied to local disk first (Drive reads are slow)
ALLY_LOCAL = '/content/ally_local'
src_files = [p for p in sorted(glob.glob(f'{ALLY_DIR}/*/*')) if p.lower().endswith(IMG_EXT)]
def _cp(p):
    d = os.path.join(ALLY_LOCAL, *p.split('/')[-2:])
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(p, d)
    return d
t0 = time.time()
with ThreadPoolExecutor(16) as ex: local_files = list(ex.map(_cp, src_files))
ALLY = [dict(path=p, cls=os.path.basename(os.path.dirname(p))) for p in local_files]
print('Allynugger: %s  (copied in %.0fs)' % (Counter(r['cls'] for r in ALLY), time.time() - t0))


In [ ]:
# CELL 7 — build crops in parallel, extract features (cached to Drive)
is_healthy = lambda c: 'healthy' in c.lower()
def _one(args):                              # worker: returns JPEG bytes of the 224x224 crop
    r, mode = args
    try:
        rgb = load_rgb(r['path']); H, W = rgb.shape[:2]
        if mode == 'box':
            b = yolo_to_box(r['yolo'], W, H); im = crop_box(rgb, b); a = (b[2] - b[0]) * (b[3] - b[1]) / (W * H)
        elif mode == 'otsu':
            im, a = otsu_crop(rgb)
        else:
            im, a = whole_crop(rgb)
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q); return buf.getvalue(), a
    except Exception as e:
        return None, str(e)

def build(records, mode):
    jobs = [(r, mode) for r in records]
    try:
        with ProcessPoolExecutor(os.cpu_count()) as ex:
            res = list(ex.map(_one, jobs, chunksize=16))
    except Exception as e:                       # e.g. pickling problems: threads still parallelise I/O
        print('  process pool unavailable (%s); using threads' % type(e).__name__)
        with ThreadPoolExecutor(8) as ex:
            res = list(ex.map(_one, jobs))
    keep = [i for i, (b, _) in enumerate(res) if b is not None]
    fails = [(records[i]['path'], res[i][1]) for i in range(len(res)) if res[i][0] is None]
    if fails: print('  %d failures, e.g. %s' % (len(fails), fails[:2]))
    crops = [Image.open(io.BytesIO(res[i][0])).convert('RGB') for i in keep]
    area = np.array([res[i][1] for i in keep]); recs = [records[i] for i in keep]
    y = np.array([0 if is_healthy(r['cls']) else 1 for r in recs]); cls = np.array([r['cls'] for r in recs])
    return crops, area, y, cls, [r['path'] for r in recs]

JOBS = [('EWU_box', EWU, 'box'), ('Ally_whole', ALLY, 'whole')] + ([('Ally_otsu', ALLY, 'otsu')] if RUN_ALLY_OTSU else [])
SETS = {}
for tag, recs, mode in JOBS:
    t0 = time.time(); cache = f'{OUT}/feat_{tag}.npz'
    crops, area, y, cls, src_ = build(recs, mode)
    if os.path.exists(cache) and len(np.load(cache)['y']) == len(y) and (np.load(cache)['y'] == y).all():
        X = np.load(cache)['X']; how = 'cached features'
    else:
        X = feats(crops); np.savez(cache, X=X, y=y, area=area, cls=cls); how = 'extracted features'
    SETS[tag] = dict(X=X, y=y, area=area, cls=cls, crops=crops, src=src_)
    print('%-11s n=%5d  healthy=%4d  diseased=%5d  median crop area %.3f  [%s, %.0fs]'
          % (tag, len(y), (y == 0).sum(), (y == 1).sum(), np.median(area), how, time.time() - t0))


In [ ]:
# CELL 8 — CONTACT SHEETS: inspect before trusting numbers
import matplotlib.pyplot as plt
for tag, S in SETS.items():
    k = min(24, len(S['crops'])); pick = np.random.default_rng(1).choice(len(S['crops']), k, replace=False)
    fig, ax = plt.subplots(3, 8, figsize=(16, 6.5))
    for a, i in zip(ax.ravel(), list(pick) + [None] * (24 - k)):
        a.axis('off')
        if i is not None: a.imshow(S['crops'][i]); a.set_title(S['cls'][i][:14], fontsize=7)
    fig.suptitle(f'{tag}: random crops fed to the model', fontsize=11); plt.tight_layout()
    plt.savefig(f'{OUT}/contact_{tag}.png', dpi=110); plt.show()
print('Check: one leaf per crop? (EWU boxes must be LEAF boxes, not lesion boxes.) '
      'For Ally_otsu, does the crop isolate the leaf or return the whole frame?')


In [ ]:
# CELL 9 — leakage check: nearest TeaLeafDiseaseBD feature for every external crop
TEA = np.concatenate([Xtr, Xte])
for tag, S in SETS.items():
    nn = np.concatenate([(S['X'][i:i + 512] @ TEA.T).max(1) for i in range(0, len(S['X']), 512)])
    S['nn'] = nn
    print('%-11s max cosine to TeaLeafDiseaseBD: median %.3f | >=0.98: %d crops | >=0.95: %d'
          % (tag, np.median(nn), (nn >= 0.98).sum(), (nn >= 0.95).sum()))
print('Any crops >= 0.98 are candidate overlaps with your own data: inspect them and exclude if identical.')


In [ ]:
# CELL 10 — evaluate the frozen probe (threshold fixed in-distribution)
def evaluate(score_fn, S, thr_list):
    res = {'auc': [], 'sens': [], 'spec': [], 'per_class_flag_rate': {}}
    per = {c: [] for c in np.unique(S['cls'])}
    for rep in range(N_REP):
        s = score_fn(rep, S); flag = s >= thr_list[rep]
        res['auc'].append(roc_auc_score(S['y'], s) if len(np.unique(S['y'])) == 2 else float('nan'))
        res['sens'].append(flag[S['y'] == 1].mean()); res['spec'].append((~flag)[S['y'] == 0].mean())
        for c in per: per[c].append(flag[S['cls'] == c].mean())
    summ = lambda a: (float(np.nanmean(a)), float(np.nanstd(a, ddof=1)))
    out = {k: summ(v) for k, v in res.items() if k != 'per_class_flag_rate'}
    out['per_class_flag_rate'] = {c: summ(v) for c, v in per.items()}
    out['n'] = int(len(S['y'])); out['n_healthy'] = int((S['y'] == 0).sum())
    return out

probe_score = lambda rep, S: 1 - PROBES[rep].predict_proba(S['X'])[:, HEALTHY_IDX]
ID = dict(X=Xte, y=dte, cls=np.array(['ID'] * len(dte)))
RESULTS = {'in_distribution_probe': evaluate(probe_score, ID, THR)}
for tag, S in SETS.items():
    RESULTS[f'{tag}_probe'] = evaluate(probe_score, S, THR)

def show(name, r):
    print('%-22s n=%5d  AUC %.3f±%.3f  sens %.3f±%.3f  spec %.3f±%.3f'
          % (name, r['n'], *r['auc'], *r['sens'], *r['spec']))
    for c, (m, s) in r['per_class_flag_rate'].items():
        if c != 'ID': print('      flagged as diseased  %-18s %.3f ± %.3f' % (c, m, s))
for k, r in RESULTS.items(): show(k, r)


In [ ]:
# CELL 11 — deployed student tea_screener.onnx (dynamic batch), verified against its export report
import onnxruntime as ort
REP_ = json.load(open(ONNX_REPORT))
sha = hashlib.sha256(open(ONNX_PATH, 'rb').read()).hexdigest()
assert sha == REP_['sha256'], 'tea_screener.onnx is not the file described in export_report.json'
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
iname = sess.get_inputs()[0].name
def onnx_scores(pils, bs=64):
    out = []
    for i in range(0, len(pils), bs):
        x = np.stack([TO_T(p).numpy() for p in pils[i:i + bs]]).astype(np.float32)
        z = sess.run(None, {iname: x})[0]; z = np.exp(z - z.max(1, keepdims=True)); out.append(z[:, 1] / z.sum(1))
    return np.concatenate(out)

# in-distribution scores: saved at export time (same crops, same order as _split.json)
s_id = np.load(ONNX_ID_SCORES); assert len(s_id) == len(dte)
a_id = roc_auc_score(dte, s_id)
print('deployed model: sha256 OK | test AUC %.4f (export report %.4f)' % (a_id, REP_['test_auc']))
assert abs(a_id - REP_['test_auc']) < 1e-4
# spot-check that THIS notebook's preprocessing gives the same scores on 64 test images
chk = np.random.default_rng(3).choice(len(TEST_PATHS), 64, replace=False)
s_chk = onnx_scores([otsu_crop(load_rgb(local(TEST_PATHS[i])))[0] for i in chk])
print('spot-check vs saved scores: median |diff| %.4f, max %.4f' % (np.median(np.abs(s_chk - s_id[chk])), np.abs(s_chk - s_id[chk]).max()))

thr_s = float(np.sort(s_id[dte == 1])[int(np.floor((1 - TARGET_SENS) * (dte == 1).sum()))])
fl = s_id >= thr_s
RESULTS['in_distribution_onnx_student'] = dict(auc=(float(a_id), 0.0), sens=(float(fl[dte == 1].mean()), 0.0),
                                               spec=(float((~fl)[dte == 0].mean()), 0.0), n=int(len(dte)),
                                               n_healthy=int((dte == 0).sum()), threshold=thr_s)
show('in_distribution_onnx', RESULTS['in_distribution_onnx_student'] | {'per_class_flag_rate': {}})
for tag, S in SETS.items():
    S['onnx'] = onnx_scores(S['crops'])
    RESULTS[f'{tag}_onnx_student'] = evaluate(lambda rep, S: S['onnx'], S, [thr_s] * N_REP)
    show(f'{tag}_onnx_student', RESULTS[f'{tag}_onnx_student'])
RESULTS['_onnx'] = dict(sha256=sha, export_rep=REP_['export_rep'], threshold_95sens=thr_s)
print('Note: ONNX rows are one model, so their ± is 0 by construction.')


In [ ]:
# CELL 12 — label-free threshold: tau = quantile of target scores at the target healthy prevalence
# (the same rule the pseudo-labelling uses; needs only an estimate of prevalence, no labels)
def ss(s, y, t):
    f = s >= t; return float(f[y == 1].mean()), float((~f)[y == 0].mean())
LABEL_FREE = {}
for tag, S in SETS.items():
    pi = float((S['y'] == 0).mean()); row = {'pi': pi}
    for dpi, name in ((0, 'true_pi'), (-0.10, 'pi_minus_10'), (0.10, 'pi_plus_10')):
        p = pi + dpi
        pr = [ss(probe_score(r, S), S['y'], np.quantile(probe_score(r, S), p)) for r in range(N_REP)]
        on = ss(S['onnx'], S['y'], np.quantile(S['onnx'], p))
        row[name] = dict(pi_used=p, probe_sens=float(np.mean([x[0] for x in pr])), probe_spec=float(np.mean([x[1] for x in pr])),
                         onnx_sens=on[0], onnx_spec=on[1])
    LABEL_FREE[tag] = row
    t = row['true_pi']
    print('%-11s pi=%.3f | probe sens %.3f spec %.3f | deployed sens %.3f spec %.3f'
          % (tag, pi, t['probe_sens'], t['probe_spec'], t['onnx_sens'], t['onnx_spec']))
RESULTS['label_free_threshold'] = LABEL_FREE


In [ ]:
# CELL 13 — local recalibration of the probe (1/2/5/10 labels per TARGET class), near-duplicates held out
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
def spec_at_sens(s, y):
    pos = np.sort(s[y == 1]); t = pos[int(np.floor((1 - TARGET_SENS) * len(pos)))]; return float((s < t)[y == 0].mean())
RECAL = {}
for tag in [t for t in SETS if t != 'Ally_otsu']:
    S = SETS[tag]; X, y, cls = S['X'], S['y'], S['cls']
    sim = X @ X.T; np.fill_diagonal(sim, -1); nn_ = sim.max(1)
    grp = connected_components(csr_matrix(X @ X.T >= DUP), directed=False)[1]
    classes_t = np.unique(cls); K = len(classes_t); cid = np.searchsorted(classes_t, cls)
    hk = int(np.where(classes_t == next(c for c in classes_t if is_healthy(c)))[0][0])
    info = dict(within_set_dup_frac=float((nn_ >= DUP).mean()), n_groups=int(grp.max() + 1),
                largest_group=int(np.bincount(grp).max()),
                healthy_in_one_group=float(np.bincount(grp[y == 0]).max() / max(1, (y == 0).sum())))
    curve = {}
    for n in (1, 2, 5, 10, 20):
        a, sp = [], []
        for rep in range(RECAL_DRAWS):
            g = np.random.default_rng(7000 + 100 * n + rep)
            lab = np.concatenate([g.choice(np.where(cid == k)[0], n, replace=False) for k in range(K)])
            te = np.where(~np.isin(grp, grp[lab]))[0]
            if len(np.unique(y[te])) < 2: continue
            m = LogisticRegression(C=1.0, max_iter=2000).fit(X[lab], cid[lab])
            s = 1 - m.predict_proba(X[te])[:, list(m.classes_).index(hk)]
            a.append(roc_auc_score(y[te], s)); sp.append(spec_at_sens(s, y[te]))
        curve[n] = dict(labels=n * K, expert_min=n * K * 8 / 60,
                        auc=(float(np.mean(a)), float(np.std(a, ddof=1))) if len(a) > 1 else None,
                        oracle_spec_at_95sens=(float(np.mean(sp)), float(np.std(sp, ddof=1))) if len(sp) > 1 else None)
    RECAL[tag] = dict(dup=info, curve=curve)
    print('%-11s within-set near-duplicates %.1f%% | groups %d | largest %d | healthy in largest group %.0f%%'
          % (tag, 100 * info['within_set_dup_frac'], info['n_groups'], info['largest_group'], 100 * info['healthy_in_one_group']))
    for n, c in curve.items():
        print('    %2d/class (%3d labels, %.1f min): AUC %s' % (n, c['labels'], c['expert_min'],
              '%.3f ± %.3f' % c['auc'] if c['auc'] else 'n/a (test set collapses after group removal)'))
    if info['within_set_dup_frac'] > 0.5:
        print('    WARNING: >50%% of %s has a near-duplicate; held-out recalibration there is not independent — do not report it' % tag)
RESULTS['recalibration'] = RECAL


In [ ]:
# CELL 14 — paper-ready summary + save
def f(v): return '%.3f ± %.3f' % tuple(v) if isinstance(v, (list, tuple)) else '%.3f' % v
rows = [('TeaLeafDiseaseBD test, probe', 'in_distribution_probe'), ('TeaLeafDiseaseBD test, deployed', 'in_distribution_onnx_student')]
for tag in SETS:
    rows += [(f'{tag}, probe', f'{tag}_probe'), (f'{tag}, deployed', f'{tag}_onnx_student')]
print('%-34s %6s %15s %15s %15s' % ('Zero-shot, threshold fixed in-dist.', 'n', 'AUC', 'sens', 'spec'))
for name, k in rows:
    r = RESULTS[k]; print('%-34s %6d %15s %15s %15s' % (name, r['n'], f(r['auc']), f(r['sens']), f(r['spec'])))
RESULTS['_meta'] = dict(target_sens=TARGET_SENS, thresholds_fixed_on='TeaLeafDiseaseBD test split',
                        probe='C=1.0, 70 labels, 5 seed draws (exact revision_v1 probes)',
                        deployed=os.path.basename(ONNX_PATH) + ' sha256 ' + RESULTS['_onnx']['sha256'][:12],
                        preprocessing='bbox crop +6% pad, 224x224, JPEG q95; EWU=YOLO leaf box; '
                                      'Ally_otsu=paper Otsu mask; Ally_whole=full frame',
                        median_crop_area={k: float(np.median(S['area'])) for k, S in SETS.items()}, dup_threshold=DUP)
json.dump(RESULTS, open(f'{OUT}/external_results_v3.json', 'w'), indent=2, default=float)
print('\nsaved ->', f'{OUT}/external_results_v3.json   (send this file + the contact_*.png images)')
